# 03. Эксперименты CP1

В формате **Гипотеза → Как проверялось → Результат**. Полный цикл из 4–5 моделей + ансамблей + hyperparameter tuning делается в CP2; здесь закрываем 7 из 18 баллов моделирования за счёт трёх сравнений с baseline.


In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold

from src.config import REPORT_IMAGES_DIR, SEED
from src.data_loader import load_raw
from src.preprocessing import time_or_random_split, build_full_pipeline, split_xy
from src.modeling import fit_and_evaluate, collect_metrics
from src.utils import set_seed, regression_metrics

set_seed()
REPORT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

df = load_raw()
train_df, val_df, test_df = time_or_random_split(df)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')


train=7000, val=1500, test=1500


## Эксперимент 1. Ridge vs LinearRegression

**Гипотеза.** L2-регуляризация уменьшит переобучение на многочисленных OHE-фичах и улучшит/сравняет качество относительно ванильной LinearRegression.

**Как проверялось.** GridSearchCV(`alpha` ∈ {0.1, 1, 10, 100}) по 5-fold CV на train, затем метрики на val и test.


In [2]:
pipe = build_full_pipeline(Ridge(random_state=SEED))
X_tr, y_tr = split_xy(train_df)
grid = GridSearchCV(pipe, {'model__alpha': [0.1, 1, 10, 100]},
                    cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_tr, y_tr)
print('Best alpha:', grid.best_params_)
print('Best CV RMSE:', -grid.best_score_)

X_va, y_va = split_xy(val_df); X_te, y_te = split_xy(test_df)
exp_rows = []
exp_rows.append({'model': f"Ridge(alpha={grid.best_params_['model__alpha']})", 'split': 'val',
                 **regression_metrics(np.asarray(y_va), grid.predict(X_va))})
exp_rows.append({'model': f"Ridge(alpha={grid.best_params_['model__alpha']})", 'split': 'test',
                 **regression_metrics(np.asarray(y_te), grid.predict(X_te))})
pd.DataFrame(exp_rows)


Best alpha: {'model__alpha': 100}
Best CV RMSE: 41089.03617261463


,model,split,RMSE,MAE,R2
0,Ridge(alpha=100),val,30739.162529,14060.074205,0.819604
1,Ridge(alpha=100),test,72868.991177,21378.078679,0.654698


## Эксперимент 2. Lasso

**Гипотеза.** L1 обнулит часть неинформативных фичей (особенно среди OHE-кодов редких категорий) и даст более компактную и интерпретируемую модель.

**Как проверялось.** GridSearchCV(`alpha` ∈ {0.01, 0.1, 1, 10}) на train.


In [3]:
pipe = build_full_pipeline(Lasso(random_state=SEED, max_iter=10000))
grid = GridSearchCV(pipe, {'model__alpha': [0.01, 0.1, 1, 10]},
                    cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_tr, y_tr)
print('Best alpha:', grid.best_params_)
print('Best CV RMSE:', -grid.best_score_)

exp_rows.append({'model': f"Lasso(alpha={grid.best_params_['model__alpha']})", 'split': 'val',
                 **regression_metrics(np.asarray(y_va), grid.predict(X_va))})
exp_rows.append({'model': f"Lasso(alpha={grid.best_params_['model__alpha']})", 'split': 'test',
                 **regression_metrics(np.asarray(y_te), grid.predict(X_te))})
pd.DataFrame(exp_rows)


/usr/local/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.387e+11, tolerance: 3.394e+09
  model = cd_fast.enet_coordinate_descent(
/usr/local/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.175e+10, tolerance: 3.394e+09
  model = cd_fast.enet_coordinate_descent(
/usr/local/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale 

/usr/local/Cellar/jupyterlab/4.4.2/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.082e+12, tolerance: 3.375e+09
  model = cd_fast.enet_coordinate_descent(


Best alpha: {'model__alpha': 10}
Best CV RMSE: 41135.17146823483


,model,split,RMSE,MAE,R2
0,Ridge(alpha=100),val,30739.162529,14060.074205,0.819604
1,Ridge(alpha=100),test,72868.991177,21378.078679,0.654698
2,Lasso(alpha=10),val,31517.089887,14660.924260,0.810358
3,Lasso(alpha=10),test,72702.396816,21881.850506,0.656275


## Эксперимент 3. KNN

**Гипотеза.** Похожие по таргетингу/креативу кампании дают похожий profit — KNN на отмасштабированных фичах может это уловить. Вероятно, проиграет линейным на многомерных OHE, но проверить полезно для контраста.

**Как проверялось.** GridSearchCV(`n_neighbors` ∈ {5, 15, 25}) на train.


In [4]:
pipe = build_full_pipeline(KNeighborsRegressor())
grid = GridSearchCV(pipe, {'model__n_neighbors': [5, 15, 25]},
                    cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_tr, y_tr)
print('Best k:', grid.best_params_)
print('Best CV RMSE:', -grid.best_score_)

exp_rows.append({'model': f"KNN(k={grid.best_params_['model__n_neighbors']})", 'split': 'val',
                 **regression_metrics(np.asarray(y_va), grid.predict(X_va))})
exp_rows.append({'model': f"KNN(k={grid.best_params_['model__n_neighbors']})", 'split': 'test',
                 **regression_metrics(np.asarray(y_te), grid.predict(X_te))})
pd.DataFrame(exp_rows)


Best k: {'model__n_neighbors': 5}
Best CV RMSE: 46883.97347891959


,model,split,RMSE,MAE,R2
0,Ridge(alpha=100),val,30739.162529,14060.074205,0.819604
1,Ridge(alpha=100),test,72868.991177,21378.078679,0.654698
2,Lasso(alpha=10),val,31517.089887,14660.924260,0.810358
3,Lasso(alpha=10),test,72702.396816,21881.850506,0.656275
4,KNN(k=5),val,40106.468360,13297.619313,0.692906
5,KNN(k=5),test,79389.434025,21424.061604,0.590137


## Эксперимент 4. RandomForest

**Гипотеза.** Нелинейная ансамблевая модель должна уловить взаимодействия между таргетингом, креативом и аукционными параметрами лучше линейных моделей.

**Как проверялось.** Фиксированные разумные гиперпараметры (`n_estimators=300, max_depth=None, min_samples_leaf=2`); подробный tuning — на CP2.


In [5]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                           random_state=SEED, n_jobs=-1)
_, rf_metrics = fit_and_evaluate(rf, train_df, val_df, test_df, 'RandomForest(300)')
exp_rows += rf_metrics
pd.DataFrame(exp_rows)


,model,split,RMSE,MAE,R2
0,Ridge(alpha=100),val,30739.162529,14060.074205,0.819604
1,Ridge(alpha=100),test,72868.991177,21378.078679,0.654698
2,Lasso(alpha=10),val,31517.089887,14660.924260,0.810358
3,Lasso(alpha=10),test,72702.396816,21881.850506,0.656275
4,KNN(k=5),val,40106.468360,13297.619313,0.692906
5,KNN(k=5),test,79389.434025,21424.061604,0.590137
6,RandomForest(300),val,41051.380657,8375.057956,0.678266
7,RandomForest(300),test,69461.758190,14869.025008,0.686235


## Сводная таблица CP1

In [6]:
exp_table = collect_metrics(exp_rows)
exp_table.to_csv(REPORT_IMAGES_DIR / 'cp1_experiments.csv', index=False)
exp_table


,model,split,RMSE,MAE,R2
0,RandomForest(300),test,69461.758190,14869.025008,0.686235
1,Lasso(alpha=10),test,72702.396816,21881.850506,0.656275
2,Ridge(alpha=100),test,72868.991177,21378.078679,0.654698
3,KNN(k=5),test,79389.434025,21424.061604,0.590137
4,Ridge(alpha=100),val,30739.162529,14060.074205,0.819604
5,Lasso(alpha=10),val,31517.089887,14660.924260,0.810358
6,KNN(k=5),val,40106.468360,13297.619313,0.692906
7,RandomForest(300),val,41051.380657,8375.057956,0.678266


## Предварительные выводы

- Ridge даёт примерно тот же или чуть лучший результат, чем LinearRegression — регуляризация не критична при текущем количестве фичей.
- Lasso проверяет гипотезу о «разреженном» сигнале.
- KNN, как правило, проигрывает на смешанных данных с OHE-признаками большой размерности.
- RandomForest — сильный кандидат в лидеры. В CP2 сравним его с градиентным бустингом.
